# VLM Disaster Analyzer — Production Colab

**Run cells top-to-bottom on a fresh GPU runtime.**  
After Cell 9 you will have a public `VITE_API_URL` ready to paste into your frontend.

| Cell | What it does |
|------|-------------|
| 1 | Clone repository + verify structure |
| 2 | Install Python dependencies + verify imports |
| 3 | GPU check — halts with instructions if no GPU |
| 4 | Verify FAISS index, metadata, and model files |
| 5 | Launch backend (non-blocking, waits for ready) |
| 6 | Create ngrok tunnel (requires free auth token) |
| 7 | Automated endpoint health checks |
| 8 | Final PASS/FAIL status report |
| 9 | Output `VITE_API_URL=...` for copy-paste |

> **Tip:** If a cell fails, fix the issue and re-run that cell only — no need to restart from Cell 1 unless instructed.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Clone repository and verify structure
#
# Clones the repository into /content/VLM-Disaster-Analyzer.
# If already cloned from a previous run, pulls latest changes instead.
# Aborts with a clear error if any required directory is missing.
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, os, sys
from pathlib import Path

REPO_URL = "https://github.com/ujjesha1312/VLM-Disaster-Analyzer"
REPO_DIR  = Path("/content/VLM-Disaster-Analyzer")

print("=" * 60)
print("  CELL 1 — Repository Setup")
print("=" * 60)
print()

# ── Clone or update ──────────────────────────────────────────────────────────
if REPO_DIR.exists():
    print("Repository already present — pulling latest changes...")
    result = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        capture_output=True, text=True,
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f"Cloning {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"git clone failed (exit {result.returncode}):\n{result.stderr}"
        )
    print("Cloned successfully.")

print()

# ── Verify required paths exist ───────────────────────────────────────────────
REQUIRED = [
    "backend",
    "backend/main.py",
    "backend/config.py",
    "backend/services/disaster_service.py",
    "backend/services/video_service.py",
    "backend/routes/predict_disaster.py",
    "backend/routes/predict_video.py",
    "backend/routes/retrieval.py",
    "src",
    "src/models/clip_model.py",
    "src/models/qwen_model.py",
    "src/retrieval/search.py",
    "datasets/historical/index",
    "datasets/historical/index/disaster.index",
    "datasets/historical/index/metadata.json",
    "requirements.txt",
    "start_backend.py",
]

print("Verifying repository structure...")
missing = []
for item in REQUIRED:
    p = REPO_DIR / item
    if p.exists():
        size = p.stat().st_size if p.is_file() else 0
        tag  = f"{size/1e6:.1f} MB" if size > 1e6 else (f"{size/1e3:.1f} KB" if size > 0 else "dir")
        print(f"  ✓  {item:<50} {tag}")
    else:
        print(f"  ✗  {item:<50} MISSING")
        missing.append(item)

print()
if missing:
    raise RuntimeError(
        f"Repository is incomplete. Missing {len(missing)} path(s):\n"
        + "\n".join(f"  - {m}" for m in missing)
        + "\n\nRe-run this cell after fixing the clone step."
    )

print("✓  Repository structure verified — all required files present.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Install dependencies and verify imports
#
# Installs requirements.txt, then adds Colab-specific extras.
# Uses faiss-gpu (preferred on T4) with faiss-cpu as fallback.
# Verifies every critical import before proceeding.
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/VLM-Disaster-Analyzer")

print("=" * 60)
print("  CELL 2 — Dependency Installation")
print("=" * 60)
print()

def pip_install(args, label, quiet=True):
    """Run pip install and return True on success."""
    cmd = [sys.executable, "-m", "pip", "install"] + args
    if quiet:
        cmd.append("-q")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✓  {label}")
        return True
    else:
        # Show last 3 error lines only to keep output clean
        err_lines = [l for l in result.stderr.splitlines() if l.strip()][-3:]
        print(f"  ⚠  {label} — issue during install:")
        for l in err_lines:
            print(f"     {l}")
        return False

# ── 1. Core requirements ──────────────────────────────────────────────────────
print("Installing requirements.txt...")
pip_install(["-r", str(REPO_DIR / "requirements.txt")], "requirements.txt")
print()

# ── 2. FAISS — prefer GPU build on Colab T4 ──────────────────────────────────
print("Installing FAISS (GPU build preferred)...")
# First try faiss-gpu; if that fails (e.g. unsupported CUDA version) fall back to CPU
ok = pip_install(["faiss-gpu"], "faiss-gpu")
if not ok:
    print("  faiss-gpu failed — falling back to faiss-cpu")
    pip_install(["faiss-cpu>=1.7.4"], "faiss-cpu (fallback)")
print()

# ── 3. Colab extras not in requirements.txt ───────────────────────────────────
print("Installing Colab-specific extras...")
extras = [
    (["pyngrok>=6.0.0"],              "pyngrok (ngrok tunnel)"),
    (["httpx>=0.27.0"],               "httpx (health checks)"),
    (["opencv-python-headless>=4.8"], "opencv-python-headless (video)"),
    (["bitsandbytes>=0.43.0"],        "bitsandbytes (Qwen 4-bit NF4)"),
]
for args, label in extras:
    pip_install(args, label)
print()

# ── 4. Verify critical imports ────────────────────────────────────────────────
print("Verifying imports...")
IMPORT_CHECKS = [
    ("torch",           "PyTorch"),
    ("transformers",    "Transformers"),
    ("fastapi",         "FastAPI"),
    ("uvicorn",         "Uvicorn"),
    ("faiss",           "FAISS"),
    ("PIL",             "Pillow"),
    ("cv2",             "OpenCV"),
    ("httpx",           "HTTPX"),
    ("pyngrok",         "pyngrok"),
    ("bitsandbytes",    "bitsandbytes"),
    ("numpy",           "NumPy"),
    ("dotenv",          "python-dotenv"),
]

failed_imports = []
for module, name in IMPORT_CHECKS:
    try:
        __import__(module)
        print(f"  ✓  {name}")
    except ImportError as e:
        print(f"  ✗  {name}: {e}")
        failed_imports.append(name)

print()
if failed_imports:
    raise RuntimeError(
        f"Import failures: {failed_imports}\n"
        "Re-run this cell. If the error persists, restart the runtime."
    )

import torch
print(f"  PyTorch version  : {torch.__version__}")
print(f"  CUDA available   : {torch.cuda.is_available()}")
print()
print("✓  All imports verified.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — GPU availability check
#
# Verifies a CUDA-capable GPU is present and reports its name and VRAM.
# Stops execution with clear instructions if no GPU is available.
# Qwen2-VL-2B-Instruct (4-bit) needs ~2.5 GB VRAM.
# CLIP-ViT-B/32 needs ~300 MB VRAM.
# Total expected: ~2.8 GB — well within a T4 (15 GB).
# ─────────────────────────────────────────────────────────────────────────────

import torch

print("=" * 60)
print("  CELL 3 — GPU Check")
print("=" * 60)
print()

if not torch.cuda.is_available():
    print("✗  No GPU detected.")
    print()
    print("  This notebook requires a GPU runtime to run Qwen2-VL-2B.")
    print("  To enable GPU:")
    print("    1. Runtime → Change runtime type")
    print("    2. Set 'Hardware accelerator' to T4 GPU")
    print("    3. Click Save")
    print("    4. Runtime → Restart session")
    print("    5. Run all cells again from Cell 1")
    raise SystemExit(
        "GPU not available. Change runtime type to T4 GPU and restart."
    )

# ── Report GPU details ────────────────────────────────────────────────────────
gpu_name   = torch.cuda.get_device_name(0)
props      = torch.cuda.get_device_properties(0)
total_vram = props.total_memory / 1e9
free_vram  = (props.total_memory - torch.cuda.memory_reserved(0)) / 1e9
cuda_ver   = torch.version.cuda

print(f"  GPU name     : {gpu_name}")
print(f"  VRAM total   : {total_vram:.1f} GB")
print(f"  VRAM free    : {free_vram:.1f} GB")
print(f"  CUDA version : {cuda_ver}")
print(f"  PyTorch      : {torch.__version__}")
print()

# ── VRAM warning ─────────────────────────────────────────────────────────────
REQUIRED_VRAM_GB = 3.5  # CLIP ~0.3 + Qwen 4-bit ~2.5 + headroom
if total_vram < REQUIRED_VRAM_GB:
    print(f"  ⚠  Only {total_vram:.1f} GB VRAM detected.")
    print(f"  Qwen2-VL-2B-Instruct (4-bit NF4) needs ~2.5 GB.")
    print(f"  The server will attempt to load on CPU instead — expect very slow inference.")
else:
    needed = REQUIRED_VRAM_GB
    headroom = total_vram - needed
    print(f"  ✓  VRAM sufficient: {needed:.1f} GB needed, {headroom:.1f} GB headroom.")

print()
print("✓  GPU check passed.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Verify files, FAISS index, metadata, and model configurations
#
# Confirms all runtime-required files are present before starting the server.
# Validates the FAISS index metadata for category coverage.
# Reports model path configuration (CLIP + Qwen are lazy-loaded on first request).
# ─────────────────────────────────────────────────────────────────────────────

import json
import sys
from pathlib import Path

REPO_DIR = Path("/content/VLM-Disaster-Analyzer")

print("=" * 60)
print("  CELL 4 — File & Index Verification")
print("=" * 60)
print()

# ── Required file manifest ────────────────────────────────────────────────────
FILE_CHECKS = {
    "FAISS index":           REPO_DIR / "datasets/historical/index/disaster.index",
    "Metadata JSON":         REPO_DIR / "datasets/historical/index/metadata.json",
    "Backend entrypoint":    REPO_DIR / "start_backend.py",
    "Backend main":          REPO_DIR / "backend/main.py",
    "Config":                REPO_DIR / "backend/config.py",
    "disaster_service":      REPO_DIR / "backend/services/disaster_service.py",
    "video_service":         REPO_DIR / "backend/services/video_service.py",
    "gpu_queue":             REPO_DIR / "backend/services/gpu_queue.py",
    "predict_disaster rt":   REPO_DIR / "backend/routes/predict_disaster.py",
    "predict_video rt":      REPO_DIR / "backend/routes/predict_video.py",
    "retrieval rt":          REPO_DIR / "backend/routes/retrieval.py",
    "CLIP model source":     REPO_DIR / "src/models/clip_model.py",
    "Qwen model source":     REPO_DIR / "src/models/qwen_model.py",
    "Retrieval search":      REPO_DIR / "src/retrieval/search.py",
    "requirements.txt":      REPO_DIR / "requirements.txt",
}

print("Checking files...")
missing_files = []
for label, path in FILE_CHECKS.items():
    if path.exists():
        size = path.stat().st_size
        if size >= 1_000_000:
            tag = f"{size/1e6:.1f} MB"
        elif size >= 1_000:
            tag = f"{size/1e3:.1f} KB"
        else:
            tag = f"{size} B"
        print(f"  ✓  {label:<30} {tag}")
    else:
        print(f"  ✗  {label:<30} MISSING: {path.relative_to(REPO_DIR)}")
        missing_files.append(label)

print()

# ── Validate FAISS metadata ───────────────────────────────────────────────────
print("Validating FAISS metadata...")
meta_path = REPO_DIR / "datasets/historical/index/metadata.json"
if meta_path.exists():
    with open(meta_path, encoding="utf-8") as f:
        meta = json.load(f)
    categories = {}
    for event in meta:
        cat = event.get("category", "unknown")
        categories[cat] = categories.get(cat, 0) + 1
    print(f"  Total events : {len(meta)}")
    for cat, count in sorted(categories.items()):
        print(f"  ✓  {cat:<20} {count} events")
    # Confirm the three supported retrieval categories exist
    for expected_cat in ("flood", "cyclone", "earthquake"):
        if expected_cat not in categories:
            print(f"  ⚠  Category '{expected_cat}' not found in metadata!")
else:
    print("  ✗  metadata.json not found — cannot validate")

print()

# ── Model configuration ───────────────────────────────────────────────────────
print("Model configuration (lazy-loaded on first inference request):")
print("  CLIP   : openai/clip-vit-base-patch32  (~300 MB VRAM, downloads if not cached)")
print("  Qwen   : Qwen/Qwen2-VL-2B-Instruct     (~2.5 GB VRAM 4-bit, downloads if not cached)")
print("  BLIP-2 : disabled in production mode")
print("  LLaVA  : disabled in production mode")
print()

if missing_files:
    raise RuntimeError(
        f"{len(missing_files)} required file(s) missing:\n"
        + "\n".join(f"  - {f}" for f in missing_files)
        + "\n\nRe-run Cell 1 to re-clone the repository."
    )

print("✓  All files verified. Ready to start the backend.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Launch backend
#
# Starts the FastAPI server as a non-blocking subprocess on port 8000.
# Polls the health endpoint every second until the server responds or times out.
# Shows rolling log output while waiting so you can track progress.
# Aborts with the last N lines of the log if startup fails.
#
# Models are NOT loaded here — they lazy-load on the first inference request.
# Expected startup time: 5–15 seconds (server only, no model loading).
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys, os, time
from pathlib import Path

REPO_DIR = Path("/content/VLM-Disaster-Analyzer")
LOG_FILE = Path("/content/backend.log")
PORT     = 8000
MAX_WAIT = 120  # seconds to wait for startup

print("=" * 60)
print("  CELL 5 — Backend Startup")
print("=" * 60)
print()

# ── Kill any existing backend from a previous run ─────────────────────────────
kill = subprocess.run(
    ["fuser", "-k", f"{PORT}/tcp"],
    capture_output=True, text=True,
)
if kill.returncode == 0:
    print(f"  Stopped previous backend on port {PORT}.")
    time.sleep(1)

# ── Clear old log ─────────────────────────────────────────────────────────────
LOG_FILE.write_text("")

# ── Environment ───────────────────────────────────────────────────────────────
env = os.environ.copy()
env["PYTHONPATH"]      = str(REPO_DIR)
env["ACTIVE_MODELS"]   = "clip,qwen"
env["ENABLE_RETRIEVAL"] = "true"
env["QUANTIZE_QWEN"]   = "true"   # 4-bit NF4 — needs bitsandbytes
env["LOG_LEVEL"]       = "INFO"

# ── Launch ─────────────────────────────────────────────────────────────────────
log_fh = open(LOG_FILE, "w")
proc = subprocess.Popen(
    [sys.executable, "start_backend.py"],
    cwd=str(REPO_DIR),
    env=env,
    stdout=log_fh,
    stderr=subprocess.STDOUT,
)

# Store PID so later cells can check it
import builtins
builtins._backend_proc = proc

print(f"  Backend PID  : {proc.pid}")
print(f"  Log file     : {LOG_FILE}")
print(f"  Port         : {PORT}")
print()
print(f"  Waiting for server to start (up to {MAX_WAIT}s)...")

# ── Poll for readiness ────────────────────────────────────────────────────────
import httpx

ready   = False
t_start = time.time()
last_log_lines_shown = 0

for tick in range(MAX_WAIT):
    time.sleep(1)

    # ── Check if the process exited unexpectedly ──────────────────────────────
    if proc.poll() is not None:
        log_fh.flush()
        log_text = LOG_FILE.read_text()
        raise RuntimeError(
            f"Backend process exited unexpectedly (code {proc.returncode}).\n"
            f"Last log output:\n{'─'*50}\n"
            + "\n".join(log_text.splitlines()[-30:])
        )

    # ── Try the health endpoint ───────────────────────────────────────────────
    try:
        r = httpx.get(f"http://localhost:{PORT}/", timeout=2)
        if r.status_code == 200:
            elapsed = time.time() - t_start
            ready = True
            print(f"\n  ✓  Server responded in {elapsed:.1f}s")
            break
    except Exception:
        pass  # not up yet

    # ── Print rolling log every 10 seconds ───────────────────────────────────
    if tick > 0 and tick % 10 == 0:
        log_fh.flush()
        all_lines = [l for l in LOG_FILE.read_text().splitlines() if l.strip()]
        new_lines  = all_lines[last_log_lines_shown:]
        if new_lines:
            for l in new_lines[-4:]:
                print(f"  {l}")
            last_log_lines_shown = len(all_lines)
        else:
            print(f"  ... waiting ({tick}s)")

if not ready:
    log_fh.flush()
    log_text = LOG_FILE.read_text()
    raise RuntimeError(
        f"Backend did not start within {MAX_WAIT}s.\n"
        f"Last log output:\n{'─'*50}\n"
        + "\n".join(log_text.splitlines()[-40:])
        + f"\n{'─'*50}\n"
        "Possible causes:\n"
        "  • Port 8000 already in use (re-run this cell to kill it)\n"
        "  • Import error in backend (check log above)\n"
        "  • Dependency missing (re-run Cell 2)\n"
    )

print(f"  Local URL    : http://localhost:{PORT}")
print(f"  API docs     : http://localhost:{PORT}/docs")
print()
print("✓  Backend is running.")
print()
print("  Note: CLIP and Qwen models load on the FIRST inference request.")
print("  First /predict/disaster call will take ~7-8 min (Qwen cold start).")
print("  Subsequent calls take ~2-5 min (warm).")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Create ngrok tunnel
#
# Exposes the local port 8000 to the internet via an ngrok HTTPS tunnel.
# Requires a free ngrok auth token — get one at:
#   https://dashboard.ngrok.com/get-started/your-authtoken
#
# The public URL is stored in os.environ["NGROK_URL"] for subsequent cells.
# Re-running this cell kills the previous tunnel and creates a new one.
# ─────────────────────────────────────────────────────────────────────────────

import os, getpass
from pyngrok import ngrok, conf

print("=" * 60)
print("  CELL 6 — ngrok Tunnel")
print("=" * 60)
print()

PORT = 8000

# ── Get auth token ────────────────────────────────────────────────────────────
# Check environment variable first so this cell is non-interactive
# if the token was pre-set (e.g. via Colab secrets).
token = os.environ.get("NGROK_AUTH_TOKEN", "").strip()

if not token:
    print("  Enter your ngrok authentication token.")
    print("  Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken")
    print()
    token = getpass.getpass("  Ngrok auth token: ").strip()

if not token:
    raise ValueError(
        "ngrok auth token is required.\n"
        "Sign up free at https://ngrok.com and paste your token above."
    )

# ── Authenticate and reset any previous tunnels ───────────────────────────────
print("  Authenticating with ngrok...")
ngrok.set_auth_token(token)

# Kill any tunnels left over from a previous cell run
try:
    ngrok.kill()
    import time; time.sleep(0.5)
except Exception:
    pass

# ── Open tunnel ───────────────────────────────────────────────────────────────
print("  Opening HTTPS tunnel to localhost:8000...")
try:
    tunnel = ngrok.connect(PORT, bind_tls=True)
    public_url = tunnel.public_url
    # Ensure HTTPS
    if public_url.startswith("http://"):
        public_url = public_url.replace("http://", "https://", 1)
except Exception as e:
    raise RuntimeError(
        f"ngrok tunnel failed: {e}\n"
        "Possible causes:\n"
        "  • Invalid auth token — double-check at https://dashboard.ngrok.com\n"
        "  • Free tier limit (1 tunnel max) — existing tunnel occupying slot\n"
        "  • Network issue in this Colab instance — try Runtime → Restart\n"
    )

# ── Store URL for downstream cells ───────────────────────────────────────────
os.environ["NGROK_URL"] = public_url

print()
print(f"  ✓  Tunnel active")
print()
print(f"  Public URL : {public_url}")
print(f"  API Docs   : {public_url}/docs")
print()
print("  This URL changes every time you re-run this cell.")
print("  After getting the final URL, update VITE_API_URL in your frontend.")
print()
print("✓  ngrok tunnel established.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Automated endpoint health checks
#
# Verifies the backend through both the local address (reliable) and the
# public ngrok URL (proves end-to-end tunnel connectivity).
#
# Checks:
#   • Root health endpoint
#   • OpenAPI docs are accessible
#   • /predict/disaster route is registered
#   • /predict/video/analyze route is registered
#   • /retrieval/status returns index_built=true
#   • FAISS event count is > 0
#   • Public URL is reachable through ngrok
# ─────────────────────────────────────────────────────────────────────────────

import os, json
import httpx

print("=" * 60)
print("  CELL 7 — Endpoint Health Checks")
print("=" * 60)
print()

LOCAL      = "http://localhost:8000"
PUBLIC_URL = os.environ.get("NGROK_URL", "")

# ngrok adds a browser warning page for non-API requests from browsers.
# The header below bypasses it for programmatic clients.
NGROK_HEADERS = {"ngrok-skip-browser-warning": "1"}

results = {}  # label → bool

def check(label, url, *, method="GET", headers=None,
          expected_status=200, body_fn=None, timeout=15):
    """Run one health check and print a PASS/FAIL line."""
    try:
        r = httpx.request(method, url, headers=headers or {}, timeout=timeout)
        ok = r.status_code == expected_status
        detail = f"HTTP {r.status_code}"
        if ok and body_fn:
            try:
                data = r.json()
                ok, extra = body_fn(data)
                if extra:
                    detail += f"  {extra}"
            except Exception as je:
                ok = False
                detail += f"  JSON parse error: {je}"
    except httpx.TimeoutException:
        ok, detail = False, "TIMEOUT"
    except Exception as e:
        ok, detail = False, str(e)[:80]

    mark = "PASS" if ok else "FAIL"
    print(f"  [{mark}]  {label:<40} {detail}")
    results[label] = ok
    return ok

# ── Local checks (always reliable) ────────────────────────────────────────────
print(f"Local checks ({LOCAL})")
print("  " + "─" * 55)

check(
    "Root health check",
    f"{LOCAL}/",
    body_fn=lambda d: (d.get("status") == "running",
                       f"status={d.get('status')}  version={d.get('version','?')}"),
)

check(
    "OpenAPI docs (/docs)",
    f"{LOCAL}/docs",
)

# OpenAPI schema — used for two route-registration checks below
try:
    schema_r = httpx.get(f"{LOCAL}/openapi.json", timeout=10)
    schema   = schema_r.json() if schema_r.status_code == 200 else {}
    paths    = schema.get("paths", {})
except Exception:
    schema, paths = {}, {}

check(
    "/predict/disaster registered",
    f"{LOCAL}/openapi.json",
    body_fn=lambda d: (
        "/predict/disaster" in d.get("paths", {}),
        "route found" if "/predict/disaster" in d.get("paths", {})
        else "ROUTE NOT FOUND in OpenAPI schema"
    ),
)

check(
    "/predict/video/analyze registered",
    f"{LOCAL}/openapi.json",
    body_fn=lambda d: (
        "/predict/video/analyze" in d.get("paths", {}),
        "route found" if "/predict/video/analyze" in d.get("paths", {})
        else "ROUTE NOT FOUND in OpenAPI schema"
    ),
)

check(
    "FAISS index loaded (/retrieval/status)",
    f"{LOCAL}/retrieval/status",
    body_fn=lambda d: (
        d.get("index_built") is True,
        f"index_built={d.get('index_built')}  events={d.get('event_count', 0)}"
    ),
)

check(
    "FAISS event count > 0",
    f"{LOCAL}/retrieval/status",
    body_fn=lambda d: (
        d.get("event_count", 0) > 0,
        f"{d.get('event_count', 0)} events in index"
    ),
)

print()

# ── Public URL check (through ngrok) ─────────────────────────────────────────
if PUBLIC_URL:
    print(f"Public URL checks ({PUBLIC_URL})")
    print("  " + "─" * 55)

    check(
        "Root via ngrok",
        f"{PUBLIC_URL}/",
        headers=NGROK_HEADERS,
        timeout=20,
        body_fn=lambda d: (d.get("status") == "running",
                           f"status={d.get('status')}"),
    )

    check(
        "Retrieval status via ngrok",
        f"{PUBLIC_URL}/retrieval/status",
        headers=NGROK_HEADERS,
        timeout=20,
        body_fn=lambda d: (
            d.get("index_built") is True,
            f"index_built={d.get('index_built')}"
        ),
    )
    print()
else:
    print("  (Skipping public URL checks — run Cell 6 to create ngrok tunnel)")
    print()

# ── Summary ───────────────────────────────────────────────────────────────────
passed = sum(results.values())
total  = len(results)
print(f"  {passed}/{total} checks passed")

failed_checks = [k for k, v in results.items() if not v]
if failed_checks:
    print()
    print("  Failed checks:")
    for fc in failed_checks:
        print(f"    ✗  {fc}")
else:
    print()
    print("✓  All health checks passed.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Final production status report
#
# Aggregates all subsystem states into a single PASS/FAIL dashboard.
# Reads the backend log to detect model configuration messages.
# This cell is safe to re-run at any time.
# ─────────────────────────────────────────────────────────────────────────────

import os, httpx
from pathlib import Path

LOCAL    = "http://localhost:8000"
LOG_FILE = Path("/content/backend.log")
NGROK_URL = os.environ.get("NGROK_URL", "")

def local_json(path, timeout=8):
    try:
        return httpx.get(f"{LOCAL}{path}", timeout=timeout).json()
    except Exception:
        return None

# ── Gather status ─────────────────────────────────────────────────────────────
# Backend
health = local_json("/")
backend_ok = health.get("status") == "running" if health else False

# FAISS
ret_data   = local_json("/retrieval/status")
faiss_ok   = ret_data.get("index_built", False) if ret_data else False
event_count = ret_data.get("event_count", 0) if ret_data else 0

# Routes from OpenAPI schema
openapi = local_json("/openapi.json")
paths   = openapi.get("paths", {}) if openapi else {}
disaster_route_ok = "/predict/disaster"      in paths
video_route_ok    = "/predict/video/analyze" in paths

# Model config from log (CLIP and Qwen are lazy-loaded — only confirm they are configured)
log_text  = LOG_FILE.read_text() if LOG_FILE.exists() else ""
qwen_configured = "qwen" in log_text.lower() or "Active models" in log_text
clip_configured = "clip" in log_text.lower() or "Active models" in log_text

# ngrok
ngrok_ok = bool(NGROK_URL)

# Retrieval categories
retrieval_ok = faiss_ok and event_count > 0

# ── Print report ──────────────────────────────────────────────────────────────
WIDTH = 62
print("=" * WIDTH)
print("  PRODUCTION STATUS REPORT — VLM Disaster Analyzer")
print("=" * WIDTH)
print()

def row(label, ok, detail=""):
    mark    = "PASS" if ok else "FAIL"
    detail_str = f"  ({detail})" if detail else ""
    print(f"  [{mark}]  {label:<30}{detail_str}")

row("Backend API",           backend_ok,        f"http://localhost:{8000}/")
row("CLIP model",            clip_configured,   "lazy-loads on first request")
row("Qwen2-VL model",        qwen_configured,   "lazy-loads on first request (~7 min cold)")
row("FAISS index",           faiss_ok,          f"{event_count} events indexed")
row("Historical retrieval",  retrieval_ok,      "flood / cyclone / earthquake")
row("/predict/disaster",     disaster_route_ok, "image pipeline endpoint")
row("/predict/video/analyze",video_route_ok,    "video pipeline endpoint")
row("ngrok tunnel",          ngrok_ok,          NGROK_URL[:52] if NGROK_URL else "run Cell 6")

print()
print("─" * WIDTH)

all_critical = all([
    backend_ok, faiss_ok, disaster_route_ok, video_route_ok, ngrok_ok,
])

if all_critical:
    print()
    print("  STATUS: ✓  READY FOR FRONTEND CONNECTION")
    print()
    print(f"  Set VITE_API_URL={NGROK_URL}")
    print("  Rebuild frontend:  cd frontend && npm run build")
else:
    blockers = []
    if not backend_ok:        blockers.append("Backend not running — re-run Cell 5")
    if not faiss_ok:          blockers.append("FAISS index not loaded — check Cell 4")
    if not disaster_route_ok: blockers.append("/predict/disaster missing — check backend/routes/")
    if not video_route_ok:    blockers.append("/predict/video/analyze missing — check backend/routes/")
    if not ngrok_ok:          blockers.append("ngrok not configured — run Cell 6")
    print()
    print("  STATUS: ✗  NOT READY — blockers:")
    for b in blockers:
        print(f"    • {b}")

print()
print("=" * WIDTH)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Output VITE_API_URL for frontend deployment
#
# Prints the one environment variable line you need to copy into your frontend.
#
# For Vercel / Netlify: paste into the Environment Variables settings UI.
# For local development: create frontend/.env.local and paste there.
# After setting the variable, rebuild the frontend:
#   cd frontend && npm run build
#
# Important: the ngrok URL changes every Colab session.
# You must re-run Cell 6 and this cell each time you restart the runtime.
# ─────────────────────────────────────────────────────────────────────────────

import os

NGROK_URL = os.environ.get("NGROK_URL", "").strip()

print("=" * 60)
print("  CELL 9 — Frontend Environment Variable")
print("=" * 60)
print()

if not NGROK_URL:
    print("  ⚠  NGROK_URL not set. Run Cell 6 first to create the tunnel.")
else:
    line = f"VITE_API_URL={NGROK_URL}"

    print("  Copy the line below into your frontend environment:")
    print()
    print("  " + "─" * 56)
    print(f"  {line}")
    print("  " + "─" * 56)
    print()
    print("  Option A — Local development (create frontend/.env.local):")
    print(f"    echo '{line}' > frontend/.env.local")
    print(f"    cd frontend && npm run build")
    print()
    print("  Option B — Vercel deployment:")
    print("    Project Settings → Environment Variables → Add")
    print(f"    Name:  VITE_API_URL")
    print(f"    Value: {NGROK_URL}")
    print("    Then redeploy.")
    print()
    print("  Option C — Netlify deployment:")
    print("    Site configuration → Environment variables → Add variable")
    print(f"    Key:   VITE_API_URL")
    print(f"    Value: {NGROK_URL}")
    print("    Then trigger a redeploy.")
    print()
    print("  ⚠  This URL expires when the Colab runtime stops.")
    print("     Re-run Cell 6 + Cell 9 each new session to get a fresh URL.")
